<a href="https://colab.research.google.com/github/jubair0614/LLMScratch/blob/main/pytorch_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

In [ ]:
print(torch.__version__)

2.11.0+cu128


In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
import numpy as np
a = torch.tensor([1, 2, 4])
b = torch.tensor([[2, 3], [5, 4]])
print(a.shape)
print(b)
print(b.transpose(1, 0))

arr = np.array([[2, 3, 4], [1, 0, 2]])
b = torch.from_numpy(arr)
print(arr)
print(torch.from_numpy(arr))

print(a.dtype)
print(b.view(3, 2))
print(b.reshape(3 ,2))
print(b.T)
print(b.matmul(b.T))
print(b @ b.T)

torch.Size([3])
tensor([[2, 3],
        [5, 4]])
tensor([[2, 5],
        [3, 4]])
[[2 3 4]
 [1 0 2]]
tensor([[2, 3, 4],
        [1, 0, 2]])
torch.int64
tensor([[2, 3],
        [4, 1],
        [0, 2]])
tensor([[2, 3],
        [4, 1],
        [0, 2]])
tensor([[2, 1],
        [3, 0],
        [4, 2]])
tensor([[29, 10],
        [10,  5]])
tensor([[29, 10],
        [10,  5]])


In [ ]:
import torch.nn.functional as F

y = torch.tensor([1.0])
w1 = torch.tensor([2.2])
x1 = torch.tensor([1.1])
b = torch.tensor([0.0])

z = w1 * x1 + b
a = F.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


In [ ]:
from torch.nn import functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = F.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [ ]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [76]:
import torch.nn as nn

class NeuralNet(nn.Module):
  def __init__(self, in_dim: int=10, out_dim: int = 3):
    super().__init__()
    self.layer1 = nn.Linear(in_dim, 6)
    self.layer2 = nn.Linear(6, 4)
    self.output = nn.Linear(4, out_dim)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.relu(self.layer1(x))
    x = self.relu(self.layer2(x))
    logits = self.output(x)
    return logits


# Changed x to have a dimension of 50 to match NeuralNetSequential's expected input
x = torch.rand(5, 10)
model = NeuralNet()
model
print(model(x))

tensor([[ 0.1760, -0.0323, -0.0603],
        [ 0.1760, -0.0323, -0.0603],
        [ 0.1760, -0.0321, -0.0605],
        [ 0.1762, -0.0318, -0.0609],
        [ 0.1793, -0.0254, -0.0678]], grad_fn=<AddmmBackward0>)


In [101]:
import torch.nn as nn
class NeuralNetSequential(nn.Module):
  def __init__(self, in_dim: int, out_dim: int):
    super().__init__()

    self.layers = nn.Sequential(
        # 1st hidden layer
        nn.Linear(in_dim, 30),
        nn.ReLU(),

        # 2nd hidden layer
        nn.Linear(30, 20),
        nn.ReLU(),

        # output layer
        nn.Linear(20, out_dim),
    )
  def forward(self, x):
    logits = self.layers(x)
    return logits

In [102]:
model = NeuralNetSequential(8, 3)
model
x = torch.rand((5, 8))
print(model(x))

tensor([[-0.1503, -0.2319, -0.1848],
        [-0.1050, -0.2121, -0.2449],
        [-0.1080, -0.2170, -0.2274],
        [-0.0750, -0.2214, -0.2054],
        [-0.0862, -0.2110, -0.2380]], grad_fn=<AddmmBackward0>)


In [103]:
model

NeuralNetSequential(
  (layers): Sequential(
    (0): Linear(in_features=8, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [ ]:
print(f"weights {8*30 + 30*20 + 20*3}, bias {30 + 20 + 3}")
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(num_params)
print(model.layers[0].weight) # however, model.layers[1].weight not correct as ReLU()
print(model.layers[0].bias)

In [105]:
print(model.layers[2].weight.shape)

torch.Size([20, 30])


In [122]:
x = torch.rand((5, 50))
model = NeuralNetSequential(50, 3)
print(model(x))
with torch.no_grad():
  prob = torch.softmax(model(x), dim=0)
print(prob)

tensor([[-0.1173,  0.1028,  0.1362],
        [-0.1162,  0.0828,  0.1453],
        [-0.0863,  0.0905,  0.1627],
        [-0.0932,  0.0599,  0.1337],
        [-0.0363,  0.0782,  0.1621]], grad_fn=<AddmmBackward0>)
tensor([[0.1945, 0.2040, 0.1976],
        [0.1947, 0.2000, 0.1994],
        [0.2006, 0.2015, 0.2030],
        [0.1993, 0.1955, 0.1971],
        [0.2109, 0.1991, 0.2028]])


In [116]:
X = torch.rand(1, 50)
model = NeuralNetSequential(50, 3)
out = model(X)
print(out)

tensor([[ 0.1509,  0.2184, -0.0804]], grad_fn=<AddmmBackward0>)


In [117]:
with torch.no_grad():
  out = model(X)
print(out)

tensor([[ 0.1509,  0.2184, -0.0804]])


In [118]:
with torch.no_grad():
  out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3492, 0.3736, 0.2771]])


In [125]:
import torch

torch.manual_seed(42)

X_train = torch.tensor([
    [-1.2, 3.1],
    [0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

In [126]:
X_test = torch.tensor([
    [0.8, 2.8],
    [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

In [135]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
  def __init__(self, X, y):
    self.features = X
    self.labels = y
  def __getitem__(self, index):
    one_x = self.features[index]
    one_y = self.labels[index]
    return one_x, one_y

  def __len__(self):
    # print(self.labels.shape)
    return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)


In [136]:
len(train_ds)

5

In [137]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 2,
    shuffle = True,
    num_workers = 0
)

In [138]:
test_loader = DataLoader(
    dataset = test_ds,
    batch_size = 2,
    shuffle = False,
    num_workers = 0
)

In [139]:
for idx, (x, y) in enumerate(train_loader):
  print(f"Batch {idx+1}", x, y)

Batch 1 tensor([[ 2.3000, -1.1000],
        [ 0.9000,  2.9000]]) tensor([1, 0])
Batch 2 tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3 tensor([[ 2.7000, -1.5000]]) tensor([1])


In [142]:
train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 2,
    shuffle = True,
    num_workers = 0,
    drop_last = True
)
for idx, (x, y) in enumerate(train_loader):
  print(f"Batch {idx+1}", x, y)

Batch 1 tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2 tensor([[ 2.3000, -1.1000],
        [ 0.9000,  2.9000]]) tensor([1, 0])


In [149]:
import torch.nn.functional as F

torch.manual_seed(123)

model = NeuralNetSequential(2, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

  model.train()

  for idx, (x, y) in enumerate (train_loader):
    logits = model(x)
    loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"{epoch+1:03d}/{num_epochs+1:03d}"
    f"| Batch {idx+1:03d}/{len(train_loader):03d}"
    f"| Train/Val loss: {loss:.2f}")

  model.eval()

001/004| Batch 001/002| Train/Val loss: 0.75
001/004| Batch 002/002| Train/Val loss: 0.65
002/004| Batch 001/002| Train/Val loss: 0.39
002/004| Batch 002/002| Train/Val loss: 0.25
003/004| Batch 001/002| Train/Val loss: 0.02
003/004| Batch 002/002| Train/Val loss: 0.00


In [150]:
model.eval()
with torch.no_grad():
  outputs = model(X_train)
print(outputs)

tensor([[ 3.2347, -4.7798],
        [ 2.3164, -3.7002],
        [ 2.5392, -3.8612],
        [-1.5550,  1.5517],
        [-1.8102,  1.8239]])


In [153]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([[0.9997, 0.0003],
        [0.9976, 0.0024],
        [0.9983, 0.0017],
        [0.0428, 0.9572],
        [0.0257, 0.9743]])
tensor([0, 0, 0, 1, 1])


In [154]:
predictions == y_train

tensor([True, True, True, True, True])

In [155]:
torch.sum(predictions == y_train)

tensor(5)

In [158]:
def compute_accuracy(model, dataloader):
  model.eval()
  correct = 0
  total_samples = 0

  for idx, (features, labels) in enumerate(dataloader):
    with torch.no_grad():
      logits = model(features)

    predictions = torch.argmax(logits, dim=1)
    compare = predictions == labels
    correct += torch.sum(compare)
    total_samples += len(compare)

  return (correct / total_samples).item()

In [159]:
compute_accuracy(model, train_loader)


1.0

In [160]:
compute_accuracy(model, test_loader)

1.0

In [161]:
torch.save(model.state_dict(), "abc_model.pth")

In [163]:
model = NeuralNetSequential(2, 2)
model.load_state_dict(torch.load("abc_model.pth", weights_only = True))

<All keys matched successfully>

Pytorch with GPU

In [2]:
import torch

tensor1 = torch.tensor([1, 5, 9])
tensor2 = torch.tensor([3, 6, 8])
print(tensor1 + tensor2)

tensor([ 4, 11, 17])


In [8]:
device = torch.cuda.is_available()
print(device)
device = torch.mps.is_available()
total_device = torch.mps.device_count()
print(f"{device}, total mps: {total_device}")

False
True, total mps: 1


In [ ]:
tensor1 = tensor1.to("mps")
tensor2 = tensor2.to("mps")
print(tensor1 + tensor2)

tensor([ 4, 11, 17], device='mps:0')


In [7]:
tensor2 = tensor2.to("cpu")
print(tensor1 + tensor2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!

In [19]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn

# Data
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

# Dataset class
class ToyDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.features = X
        self.labels = y
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

# dataloader
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=0)

# Neural Net
class DummyNeuralNet(nn.Module):
    def __init__(self, in_dim: int = 4, out_dim: int = 2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_dim, 30),
            nn.ReLU(),

            nn.Linear(30, 15),
            nn.ReLU(),

            nn.Linear(15, 2)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


# experiment

torch.manual_seed(42)

model = DummyNeuralNet(2, 2)
device = torch.device("mps" if torch.mps.is_available() else "cpu")

model.to(device)
# optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
num_epochs = 3

for ep in range(num_epochs):
    model.train()
    for idx, (features, labels) in enumerate(train_loader):

        features, labels = features.to(device), labels.to(device)
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch {ep+1:03d}/{num_epochs:03d} | batch {idx+1:03d}/{len(train_loader):03d} | train loss: {loss:0.2f}")


Epoch 001/003 | batch 001/002 | train loss: 0.60
Epoch 001/003 | batch 002/002 | train loss: 0.64
Epoch 002/003 | batch 001/002 | train loss: 0.29
Epoch 002/003 | batch 002/002 | train loss: 0.11
Epoch 003/003 | batch 001/002 | train loss: 0.02
Epoch 003/003 | batch 002/002 | train loss: 0.00


In [23]:
# compute accuracy
def compute_accuracy(model, dataloader, device):
    model.eval()
    correct_sample = 0.0
    total_sample = 0
    print(len(dataloader))

    for idx, (features, labels) in enumerate(dataloader):
        features, labels = features.to(device), labels.to(device)
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = predictions == labels
        correct_sample += torch.sum(compare)
        total_sample += len(compare)

    print(f"Accuracy: {correct_sample / total_sample:0.4f}")
    return (correct_sample / total_sample).item()

compute_accuracy(model, train_loader, device=device)

2
Accuracy: 1.0000


1.0

In [24]:
compute_accuracy(model, test_loader, device=device)

1
Accuracy: 1.0000


1.0